In [46]:
import heapq
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import itertools
import tqdm

In [47]:
pretty_template = go.layout.Template(
    layout=dict(
        font=dict(family="iosevka"), 
        margin=dict(t=30, b=30, l=30, r=30),
    )
)

pio.templates["pretty"] = pretty_template
pio.templates.default = "plotly+pretty"

In [48]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.ones_like(priors, dtype=float) / len(priors)
    return priors / np.sum(priors)

def best_response(x, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for i, x_p in enumerate(search_space):
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost + ((len(search_space)-i)*1e-6))
    return search_space[np.argmax(utilities)]

def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors).flatten()
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump], axis=1)
    best_idx = np.argmax(utilities + np.array([(utilities.shape[1] - i) * 1e-6 for i in range(utilities.shape[1])]), axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])
    return X_p

def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true, return_breakdowns=False):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    if return_breakdowns:
        return np.dot(losses, posteriors), losses * priors
    else:
        return np.dot(losses, posteriors)

def evaluate_partition(X, partition, thresholds, priors, threshold_true, c, return_breakdowns=False):
    thresholds_p = thresholds[partition]
    priors_p     = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    return accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true, return_breakdowns)

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c, return_breakdowns=False):
    acc_loss = 0.0
    acc_loss_list = []
    for partition in partitions:
        if return_breakdowns:
            acc_loss_p, acc_loss_p_list = evaluate_partition(X, partition, thresholds, priors, threshold_true, c, return_breakdowns)
            acc_loss_list.append(acc_loss_p_list.tolist())
        else:
            acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c, return_breakdowns)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    if return_breakdowns:
        return acc_loss, acc_loss_list
    else:
        return acc_loss

In [49]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return
    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    best_partition, best_loss = None, np.inf
    for partition in set_partitions(indices):
        acc_loss = evaluate_system(X, partition, thresholds, priors, threshold_true, c)
        if acc_loss <= best_loss:
            if best_loss - acc_loss < 1e-9:
                if len(partition) < len(best_partition):
                    best_loss = acc_loss
                    best_partition = partition
            else:
                best_loss = acc_loss
                best_partition = partition
    
    return best_partition

def display_priority_queue(pq, P):
    res = "[  "
    for acc_loss, (a_id, b_id) in pq:
        res += f"({-acc_loss:.4e}, ({P[a_id]}, {P[b_id]}))  "
    res += "]"
    print(res)

def find_partitions_greedy_agg(X, thresholds, priors, threshold_true, c, eps=1e-9, show_steps=False):
    X_eps = np.arange(-1/c, 0, 1e-3).round(5)
    
    def block_cost(block):
        return (evaluate_partition(X, block, thresholds, priors, threshold_true, c)
                + evaluate_partition(X_eps, block, thresholds, priors, threshold_true, c) * eps)
    
    P = {}
    next_id = 0
    for i in range(len(priors)):
        P[next_id] = [i]
        next_id += 1

    pq = []
    for a_id, b_id in itertools.combinations(P.keys(), 2):
        a, b = P[a_id], P[b_id]
        ab = sorted(a+b)
        acc_loss_ab = block_cost(ab) * np.sum(priors[ab])
        acc_loss_a  = block_cost(a)  * np.sum(priors[a])
        acc_loss_b  = block_cost(b)  * np.sum(priors[b])
        gain = -(acc_loss_a + acc_loss_b - acc_loss_ab)
        heapq.heappush(pq, (gain, (a_id, b_id)))

    while pq:
        if show_steps:
            display_priority_queue(pq, P)
        gain, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P or b_id not in P:
            continue
        a, b = P[a_id], P[b_id]
        ab = sorted(a + b)

        acc_loss_a  = block_cost(a)
        acc_loss_b  = block_cost(b)
        acc_loss_ab = block_cost(ab)

        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-9:
            del P[a_id]
            del P[b_id]
            pq = [(g, (x, y)) for g, (x, y) in pq if x not in {a_id, b_id} and y not in {a_id, b_id}]
            heapq.heapify(pq)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P:
                if p_id == new_id:
                    continue
                p = P[p_id]
                merged = sorted(ab + p)
                acc_loss_merged = block_cost(merged) * np.sum(priors[merged])
                acc_loss_p      = block_cost(p) * np.sum(priors[p])
                acc_loss_ab_new = block_cost(ab) * np.sum(priors[ab])
                gain = -(acc_loss_p + acc_loss_ab_new - acc_loss_merged)
                heapq.heappush(pq, (gain, (new_id, p_id)))
    return list(P.values())


def split_partition(partition):
    idx_large = 0
    block_large = partition[idx_large]
    block_others = [partition[i] for i in range(len(partition)) if i != idx_large]
    result = []
    for part in itertools.combinations(block_large, len(block_large)-1):
        A = list(part)
        B = [x for x in block_large if x not in A]
        result.append([A] + [B] + block_others)

        for i, block in enumerate(block_others):
            merged = sorted(B + block)
            other_remaining = [block_others[j] for j in range(len(block_others)) if j != i]
            result.append([A] + [merged] + other_remaining)
    return result

def display_priority_queue_div(pq):
    res = "[  "
    for acc_loss, partition in pq:
        res += f"({acc_loss:.4e}, {partition})  "
    res += "]"
    print(res)

def find_partitions_greedy_div(X, thresholds, priors, threshold_true, c, eps=1e-9, show_steps=False):
    n = len(priors)
    X_eps = np.arange(-1/c, 0, 1e-3).round(5)
    
    def block_cost(block):
        return (evaluate_partition(X, block, thresholds, priors, threshold_true, c)
                + evaluate_partition(X_eps, block, thresholds, priors, threshold_true, c) * eps)

    partition_0 = [list(range(n))]
    acc_loss_0 = block_cost(partition_0[0])
    pq = [(acc_loss_0, partition_0)]

    while pq:
        if show_steps:
            display_priority_queue_div(pq)
        acc_loss_merged, partition_merged = heapq.heappop(pq)
        if len(partition_merged[0])==1:
            return partition_merged
        partitions = split_partition(partition_merged)
        pq = []
        for partition in partitions:
            acc_loss_split = 0.
            for block in partition:
                acc_loss_split += block_cost(block) * np.sum(priors[block])
            gain = acc_loss_merged - acc_loss_split
            if gain > -1e-9:
                heapq.heappush(pq, (acc_loss_split, partition))
        if not pq:
            return partition_merged

def approximation_ratio(loss_optimal, loss_greedy, rtype="m"):
    if rtype in ["a", "add", "additive"]:
        return loss_greedy - loss_optimal
    elif rtype in ["m", "mult", "multiplicative"]:
        if loss_optimal == 0:
            return np.nan
        return loss_greedy / loss_optimal

In [50]:
def run_example(X, thresholds, priors, tt, c, show_steps=False):
    base = [[i] for i in range(len(thresholds))]
    opt = find_partitions_optimal(X, thresholds, priors, tt, c)
    if show_steps:
        print("Forward")
    greedy_agg = find_partitions_greedy_agg(X, thresholds, priors, tt, c, show_steps=show_steps)
    if show_steps:
        print("\nReverse")
    greedy_div = find_partitions_greedy_div(X, thresholds, priors, tt, c, show_steps=show_steps)
    if show_steps:
        print()
    loss_base, loss_list_base = evaluate_system(X, base, thresholds, priors, tt, c, return_breakdowns=True)
    loss_opt, loss_list_opt   = evaluate_system(X, opt, thresholds, priors, tt, c, return_breakdowns=True)
    loss_greedy_agg, loss_list_greedy_agg = evaluate_system(X, greedy_agg, thresholds, priors, tt, c, return_breakdowns=True)
    loss_greedy_div, loss_list_greedy_div = evaluate_system(X, greedy_div, thresholds, priors, tt, c, return_breakdowns=True)
    r_agg = approximation_ratio(loss_opt, loss_greedy_agg)
    r_div = approximation_ratio(loss_opt, loss_greedy_div)
    r_min = min(r_agg, r_div)
    
    return {
        "thresholds": thresholds, "priors": priors, "tt": tt, "c": c,
        "base": base, "loss_base": loss_base, "loss_list_base": loss_list_base,
        "opt": opt, "loss_opt": loss_opt, "loss_list_opt": loss_list_opt,
        "greedy_agg": greedy_agg, "loss_greedy_agg": loss_greedy_agg, "loss_list_greedy_agg": loss_list_greedy_agg, "r_agg": r_agg,
        "greedy_div": greedy_div, "loss_greedy_div": loss_greedy_div, "loss_list_greedy_div": loss_list_greedy_div, "r_div": r_div,
        "r_min": r_min
        }

In [51]:
def print_partition_loss(partition, loss_list, loss, r=None):
    res_partition = "Partition          : ["
    res_loss_list = "Partition loss     : ["
    res_loss      = f"Loss               : {loss:.7f}"
    for i in range(len(partition)):
        part = partition[i]
        loss = loss_list[i]
        res_partition += "["
        res_loss_list += "["
        for p, l in zip(part, loss):
            res_partition += f"{p}".center(11)
            res_loss_list += f"{l:.7f}".center(11)
        res_partition += "]"
        res_loss_list += "]"
    res_partition += "]"
    res_loss_list += "]"
    res = res_partition + "\n" + res_loss_list + "\n" + res_loss + "\n"
    if r:
        res_r         = f"Approx ratio       : {r:.7f}"
        res += res_r + "\n"
    print(res)

In [52]:
def display_run_result(result, include_base=True):
    line_width = 73 - 6*(1-include_base)
    print(f"c                  : {result["c"]}")
    print(f"t*                 : {result["tt"]:.4f}")
    with np.printoptions(formatter={'float': '{: 0.6f}'.format}):
        print(f"priors             : {result["priors"]}")
        print(f"thresholds         : {result["thresholds"]}\n")
    if include_base:
        print(f"Baseline\n{"-"*line_width}")
        print_partition_loss(result["base"], result["loss_list_base"], result["loss_base"])
    print(f"Optimal\n{"-"*line_width}")
    print_partition_loss(result["opt"], result["loss_list_opt"], result["loss_opt"])
    print(f"Greedy (Forward)\n{"-"*line_width}")
    print_partition_loss(result["greedy_agg"], result["loss_list_greedy_agg"], result["loss_greedy_agg"], result["r_agg"])
    print(f"Greedy (Reverse)\n{"-"*line_width}")
    print_partition_loss(result["greedy_div"], result["loss_list_greedy_div"], result["loss_greedy_div"], result["r_div"])
    print(f"Approx ratio (min) : {result["r_min"]:.7f}")

In [53]:
def display_partition_result(X, thresholds, priors, c, tt, a, b):
    print(f"c                  : {c}")
    print(f"t*                 : {tt:.4f}")
    with np.printoptions(formatter={'float': '{: 0.4f}'.format}):
        print(f"priors             : {priors}")
        print(f"thresholds         : {thresholds}\n")
    
    ab = np.unique(sorted(a+b)).tolist()
    if a:
        loss_a, loss_list_a   = evaluate_partition(X, a, thresholds, priors, tt, c, return_breakdowns=True)
        loss_a = loss_a * np.sum(priors[a])
    if b:
        loss_b, loss_list_b   = evaluate_partition(X, b, thresholds, priors, tt, c, return_breakdowns=True)
        loss_b = loss_b * np.sum(priors[b])
    if a and b:
        loss_ab, loss_list_ab = evaluate_partition(X, ab, thresholds, priors, tt, c, return_breakdowns=True)
        loss_ab = loss_ab * np.sum(priors[ab])
        gain = loss_a + loss_b - loss_ab
        
        print_partition_loss([a, b], [loss_list_a, loss_list_b], loss_a + loss_b)
        print_partition_loss([ab], [loss_list_ab], loss_ab)
        print(f"Gain (merge?)      : {gain:.6f} ({gain > -1e-6})")
    else:
        if a:
            print_partition_loss([a], [loss_list_a], loss_a)
        elif b:
            print_partition_loss([b], [loss_list_b], loss_b)

In [103]:
X = np.arange(0., 1. + 1e-4, 1e-4).round(4)
# X = np.linspace(0., 1., 21).round(4)
thresholds = np.linspace(0.0, 1.0, 5)
len(X)

10001

In [81]:
# Example 1
p1 = 0.02/0.98
p2 = 0.02/0.98
diff = 1 - p1 - p2
priors = np.array([0.14 * diff/0.94, p1, p2, 0.56 * diff/0.94, 0.24 * diff/0.94])
assert(abs(np.sum(priors) - 1) < 1e-6)
c = 1.
tt = 0.1489

In [101]:
# Example 2
priors = np.array([0.097, 0.015, 0.015, 0.6286, 0.2444])
p = (0.2444/0.25 - 0.97) / (0.4888/0.25 - 0.97)
diff = 1-2*p
priors = np.array([0.097*diff/0.97, p, p, 0.6286*diff/0.97, 0.2444*diff/0.97])
assert(abs(np.sum(priors) - 1) < 1e-6)
c = 1.
tt = 0.1

In [57]:
# # Example 3
# priors = np.array([0.049, 0.01, 0.01, 0.685, 0.246])
# assert(abs(np.sum(priors) - 1) < 1e-6)
# c = 1.
# tt = 0.05

In [102]:
# m = 21
# n = 9
# thresholds = np.linspace(0.0, 1.0, n)
# priors = np.full(n, 1/n)
# c = 1.
# tt = 0.1

# X = np.linspace(0., 1., m).round(4)

result = run_example(X, thresholds, priors, tt, c, show_steps=False)
display_run_result(result, include_base=False)

c                  : 1.0
t*                 : 0.1000
priors             : [ 0.098457  0.007714  0.007714  0.638043  0.248071]
thresholds         : [ 0.000000  0.250000  0.500000  0.750000  1.000000]

Optimal
-------------------------------------------------------------------
Partition          : [[     1          2     ][     0          3          4     ]]
Partition loss     : [[ 0.0007714  0.0007714 ][ 0.0098457  0.0000032  0.0000012 ]]
Loss               : 0.0113929

Greedy (Forward)
-------------------------------------------------------------------
Partition          : [[     0          1     ][     2          3          4     ]]
Partition loss     : [[ 0.0098457  0.0005966 ][ 0.0007714  0.0638008  0.0248058 ]]
Loss               : 0.0998203
Approx ratio       : 8.7616020

Greedy (Reverse)
-------------------------------------------------------------------
Partition          : [[     2          3          4     ][     0          1     ]]
Partition loss     : [[ 0.0007714  0.0638008

In [59]:
a, b = [0,3], [4]
ab = np.unique(sorted(a+b)).tolist()

display_partition_result(X, thresholds, priors, c, tt, a, b)

X_p = best_response_vectorized(X, thresholds[ab], priors[ab], c)
fig = px.scatter(x=X, y=X_p, labels={"x": "X", "y": "BR"})
fig.add_vline(x=tt, annotation=dict(text=f"t*: {tt:.4f}",y=0.1, font=dict(color="red")), line=dict(dash="dash", color="red"))
fig.update_layout(width=500, height=400, yaxis=dict(range=[-0.05, 1.05]))

c                  : 1.0
t*                 : 0.1000
priors             : [ 0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111]
thresholds         : [ 0.0000  0.1250  0.2500  0.3750  0.5000  0.6250  0.7500  0.8750  1.0000]

Partition          : [[     0          3     ][     4     ]]
Partition loss     : [[ 0.0105820  0.0105820 ][ 0.0105820 ]]
Loss               : 0.0317460

Partition          : [[     0          3          4     ]]
Partition loss     : [[ 0.0105820  0.0105820  0.0105820 ]]
Loss               : 0.0317460

Gain (merge?)      : 0.000000 (True)


In [62]:
thresholds = np.linspace(0.0, 1.0, 5)

def test(tt,m,p4):
    p1=p2=m/2; p0=tt*(1-m); p3=1-p0-m-p4
    pr=np.array([p0,p1,p2,p3,p4])
    if np.any(pr<0): return None
    result = run_example(X, thresholds, pr, tt, c)
    opt, loss_opt = result["opt"], result["loss_opt"]
    greedy_agg, loss_greedy_agg = result["greedy_agg"], result["loss_greedy_agg"]
    greedy_div, loss_greedy_div = result["greedy_div"], result["loss_greedy_div"]
    
    r=min(loss_greedy_agg,loss_greedy_div) / loss_opt
    return r

In [98]:
X=np.linspace(0,1,200001)
tt=0.005
lo=0.0003 
hi=0.0050
peak=(0,None)
for m in np.round(np.arange(lo,hi,0.0001),5):
    r=test(tt,m,0.25*(1-0.75*m))
    if r > peak[0]: 
        peak=(r,m)
print(f"tau={tt}: peak ratio={peak[0]:.2f} at m={peak[1]}")

tau=0.005: peak ratio=108.77 at m=0.0032


In [99]:
r, m = peak
p0 = tt*(1-m)
p1 = p2 = m/2
p4 = 0.25*(1 - 0.75*m)
p3 =1 - p0 - m - p4

priors = np.array([p0,p1,p2,p3,p4])
results = run_example(X, thresholds, priors, tt, c)

In [100]:
display_run_result(results, include_base=False)

c                  : 1.0
t*                 : 0.0050
priors             : [ 0.004984  0.001600  0.001600  0.742416  0.249400]
thresholds         : [ 0.000000  0.250000  0.500000  0.750000  1.000000]

Optimal
-------------------------------------------------------------------
Partition          : [[     1          2     ][     0          3          4     ]]
Partition loss     : [[ 0.0000080  0.0000080 ][ 0.0000249  0.0000037  0.0000012 ]]
Loss               : 0.0000459

Greedy (Forward)
-------------------------------------------------------------------
Partition          : [[     0          1     ][     2          3          4     ]]
Partition loss     : [[ 0.0000249  0.0000032 ][ 0.0000080  0.0037083  0.0012457 ]]
Loss               : 0.0049902
Approx ratio       : 108.7688968

Greedy (Reverse)
-------------------------------------------------------------------
Partition          : [[     2          3          4     ][     0          1     ]]
Partition loss     : [[ 0.0000080  0.00370

In [856]:
a, b = [0], [1]
ab = np.unique(sorted(a+b)).tolist()

display_partition_result(X, thresholds, priors, c, tt, a, b)

X_im = X[::5]
X_p = best_response_vectorized(X_im, thresholds[ab], priors[ab], c)
fig = px.scatter(x=X_im, y=X_p, labels={"x": "X", "y": "BR"})
fig.add_vline(x=tt, annotation=dict(text=f"t*: {tt:.4f}",y=0.1, font=dict(color="red")), line=dict(dash="dash", color="red"))
fig.update_layout(width=500, height=400, yaxis=dict(range=[-0.05, 1.05]))

c                  : 1.0
t*                 : 0.0100
priors             : [ 0.0099  0.0030  0.0030  0.7352  0.2489]
thresholds         : [ 0.0000  0.2500  0.5000  0.7500  1.0000]

Partition          : [[     0     ][     1     ]]
Partition loss     : [[ 0.0000994 ][ 0.0000300 ]]
Loss               : 0.0001294

Partition          : [[     0          1     ]]
Partition loss     : [[ 0.0000994  0.0000245 ]]
Loss               : 0.0001239

Gain (merge?)      : 0.000006 (True)
